# Thetis — Treino ProtoNet Few-Shot (RGB, 5-way 5-shot) no Colab

Meta-treino episódico do baseline **ProtoNet** com backbone **R(2+1)D-18** sobre o
dataset [THETIS](https://github.com/THETIS-dataset/dataset), no protocolo N-way K-shot.

**Antes de começar:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU`.
- **L4** — melhor custo/velocidade em unidades (recomendado).
- **A100** — mais rápida; queima mais unidades.
- **T4** — funciona, mas é a mais lenta.

O gargalo deste treino é o **decode de vídeo**, não a GPU: a amostragem episódica
reusa o mesmo pool de ~990 clipes milhares de vezes. O `ThetisDataset` já tem um
**cache de decode** (decodifica cada clipe uma vez, redimensiona e serve da RAM).
Este notebook descompacta o `VIDEO_RGB.zip` no disco local do Colab, então cada
clipe é lido uma única vez e o treino fica *compute-bound* — e aí a GPU rende.

## 0. Verificar a GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader \
    || print("SEM GPU — ative em Ambiente de execução > Alterar o tipo de ambiente > GPU")
import torch
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")

zsh:1: command not found: nvidia-smi
zsh:1: unknown file attribute:  
torch 2.12.0 | CUDA disponível: False


## 1. Configuração

Ajuste os caminhos abaixo. **Suba para o seu Google Drive** (uma vez):

```
<Drive>/Thetis_data/
├── VIDEO_RGB.zip          ← zip da pasta VIDEO_RGB (~3.7 GB)
│                             gere com:  cd dataset && zip -r -0 VIDEO_RGB.zip VIDEO_RGB
├── manifest.csv           ← copie de data/processed/manifest.csv
└── excluded_clips.json    ← copie de data/processed/excluded_clips.json
```

> Os dois arquivos de `data/processed/` saem do `make preprocess` local. O
> `excluded_clips.json` precisa ficar **na mesma pasta** do `manifest.csv`: é ali
> que o treino procura, e sem ele o filtro de clipes defeituosos não é aplicado
> (o log avisa). Se você regerar o manifesto, regere os dois juntos.

> O código vem do GitHub; só os **dados** ficam no Drive (para persistir e não
> re-subir a cada sessão). O `VIDEO_RGB.zip` é descompactado para o disco local
> do Colab (`/content/dataset`), o que deixa até a 1ª época rápida. Os
> checkpoints/logs são gravados no Drive, então sobrevivem ao fim da sessão.


In [ ]:
REPO_URL  = "https://github.com/CauBitten/thetis.git"
REPO_DIR  = "/content/Thetis"

# Raiz dos dados no seu Google Drive (suba VIDEO_RGB.zip e manifest.csv aqui)
DRIVE_ROOT    = "/content/drive/MyDrive/Thetis_data"
RGB_ZIP       = f"{DRIVE_ROOT}/VIDEO_RGB.zip"   # zip contendo a pasta VIDEO_RGB/
MANIFEST_PATH = f"{DRIVE_ROOT}/manifest.csv"
LOCAL_DATASET = "/content/dataset"              # destino do unzip (disco local, rápido)
# Fallback: se você já tiver a pasta descompactada no Drive em vez do zip
DRIVE_DATASET_DIR = f"{DRIVE_ROOT}/dataset"

# Saídas persistidas no Drive (sobrevivem ao fim da sessão)
OUTPUT_ROOT = f"{DRIVE_ROOT}/outputs"
LOG_ROOT    = f"{DRIVE_ROOT}/logs"

BASE_CONFIG = "experiments/configs/protonet_rgb_5w5s.yaml"
RUN_ID      = "protonet_rgb_5w5s_colab"

## 2. Montar Drive, clonar o repositório e instalar dependências

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
# O Colab já traz torch(CUDA)/torchvision/numpy/pandas/opencv/scipy/matplotlib/pyyaml.
# Falta só o decord (decode de vídeo rápido no Linux — usado automaticamente pelo loader).
!pip install -q decord
import importlib
faltando = []
for m in ["torch","torchvision","cv2","pandas","numpy","yaml","scipy","matplotlib","decord"]:
    try:
        importlib.import_module(m)
    except Exception as e:
        faltando.append((m, str(e)))
print("OK — todas presentes" if not faltando else f"FALTANDO: {faltando}")

## 3. Descompactar e conferir os dados

Descompacta `VIDEO_RGB.zip` do Drive para o disco local do Colab (`/content/dataset`)
e valida os caminhos. Reexecutar é seguro: se já estiver descompactado, ele pula.

In [ ]:
from pathlib import Path
import time

def _has_rgb(root):
    return Path(root, "VIDEO_RGB").is_dir()

if _has_rgb(LOCAL_DATASET):
    DATASET_ROOT = LOCAL_DATASET
    print(f"VIDEO_RGB já descompactado em {DATASET_ROOT}")
elif Path(RGB_ZIP).is_file():
    print(f"Descompactando {RGB_ZIP}\n            -> {LOCAL_DATASET} (pode levar alguns minutos)...")
    Path(LOCAL_DATASET).mkdir(parents=True, exist_ok=True)
    t = time.time()
    !unzip -q -n "{RGB_ZIP}" -d "{LOCAL_DATASET}"
    DATASET_ROOT = LOCAL_DATASET
    print(f"  ok em {time.time()-t:.0f}s")
elif _has_rgb(DRIVE_DATASET_DIR):
    DATASET_ROOT = DRIVE_DATASET_DIR   # fallback: pasta já descompactada no Drive (mais lento)
    print(f"usando a pasta já no Drive: {DATASET_ROOT}")
else:
    raise FileNotFoundError(
        f"Não achei nem {RGB_ZIP} nem {DRIVE_DATASET_DIR}/VIDEO_RGB. "
        f"Confira DRIVE_ROOT e se o upload para o Drive terminou."
    )

assert _has_rgb(DATASET_ROOT), f"VIDEO_RGB não encontrado em {DATASET_ROOT}"
assert Path(MANIFEST_PATH).is_file(), f"manifest.csv não encontrado em {MANIFEST_PATH}"
n_avi = sum(1 for _ in Path(DATASET_ROOT, "VIDEO_RGB").rglob("*.avi"))
print(f"OK - {n_avi} vídeos em {DATASET_ROOT}/VIDEO_RGB  |  manifest: {MANIFEST_PATH}")

_exc = Path(MANIFEST_PATH).parent / "excluded_clips.json"
if _exc.is_file():
    import json
    print(f'exclusoes: {json.loads(_exc.read_text())["counts"]["rgb"]} clipes rgb filtrados')
else:
    print(f"AVISO: {_exc} nao existe -- o treino roda SEM filtrar clipes defeituosos.")
    print("       Copie data/processed/excluded_clips.json para junto do manifest.csv no Drive.")

> **Não tem o `manifest.csv`?** Se você tiver a árvore completa `dataset/VIDEO_*` no
> Drive, construa-o uma vez com a célula abaixo (descomente) e aponte `MANIFEST_PATH`
> para `data/processed/manifest.csv`.

In [ ]:
# Fallback: se você tiver a árvore dataset/VIDEO_* completa e não os arquivos do
# Drive, gere manifesto + lista de exclusões aqui (descomente). O --full-integrity
# é o que produz o excluded_clips.json.
# !python src/data/loader.py --input {DATASET_ROOT} --output data --seed 42 --full-integrity
# MANIFEST_PATH = str(Path("data/processed/manifest.csv").resolve())
# print("manifest →", MANIFEST_PATH)


## 4. Ajustar a config para o Colab

Gera uma config derivada da baseline ajustando **só o que é específico da
máquina**: caminhos dos dados e destinos de checkpoint/log no Drive.

> `encoder.batch_size` **não** é ajustado pela VRAM. O `ProtoNet._encode` fatia o
> lote em chunks desse tamanho e o R(2+1)D-18 tem 37 `BatchNorm3d` que, em modo
> treino, normalizam por chunk — mudar o valor muda o resultado, não só a
> memória. Ele vem do config e é o mesmo em todas as modalidades. Se faltar VRAM,
> use `GRAD_CHECKPOINT` (não altera o resultado). Detalhes em
> `experiments/configs/README.md`.

In [ ]:
import yaml, torch
from pathlib import Path

cfg = yaml.safe_load(Path(BASE_CONFIG).read_text())

cfg["data"]["manifest_path"] = str(Path(MANIFEST_PATH).resolve())
cfg["data"]["dataset_root"]  = str(Path(DATASET_ROOT).resolve())

cfg["output_root"] = str(Path(OUTPUT_ROOT).resolve())
cfg["log_root"]    = str(Path(LOG_ROOT).resolve())
cfg["run_id"]      = RUN_ID

COLAB_CONFIG = "experiments/configs/_colab_active.yaml"
Path(COLAB_CONFIG).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

vram = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
bs = cfg["encoder"]["batch_size"]
print(f"VRAM={vram:.1f} GB | encoder.batch_size={bs} (do config, NAO auto-escalado)")
if vram and vram < 12:
    print(f"  AVISO: batch_size={bs} pode dar OOM em {vram:.1f} GB. Prefira uma GPU maior a")
    print("  baixar o valor -- baixar muda o regime de BatchNorm e invalida a comparacao.")
print()
print(Path(COLAB_CONFIG).read_text())

## 5. Teste de sanidade (smoke)

Roda o pipeline ponta-a-ponta em segundos (encoder aleatório, 1 época, dims mínimas).
Serve para pegar erro de caminho/dados **antes** do treino longo.

In [ ]:
!python src/training/meta_trainer.py --config {COLAB_CONFIG} --smoke

## 6. Treino completo

100 épocas × 200 episódios. Procure no log a linha `[setup] decode_cache=True ...`:
a **1ª época** decodifica cada clipe uma vez (mais lenta, sobretudo lendo do Drive);
da 2ª em diante os clipes vêm da RAM. O melhor checkpoint é salvo em
`OUTPUT_ROOT/checkpoints/RUN_ID/best.pt` sempre que a acurácia de validação melhora
— então mesmo que a sessão caia, você mantém o melhor modelo.

In [ ]:
!python src/training/meta_trainer.py --config {COLAB_CONFIG}

## 7. Avaliação no meta_test (1000 episódios)

In [ ]:
from pathlib import Path
ckpt = f"{OUTPUT_ROOT}/checkpoints/{RUN_ID}/best.pt"
assert Path(ckpt).exists(), f"checkpoint não encontrado: {ckpt} (o treino terminou?)"
!python src/training/eval_episodic.py --checkpoint "{ckpt}" --output-root "{OUTPUT_ROOT}"

## 8. Curvas de treino (acurácia + loss) e matriz de confusão

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

log = json.loads(Path(f"{LOG_ROOT}/{RUN_ID}/training.json").read_text())
E = log["epochs"]
ep        = [e["epoch"] for e in E]
train_acc = [e["train_acc"] for e in E]
train_los = [e["train_loss"] for e in E]
# val_acc e val_loss só existem nas épocas de eval; mantidos alinhados por época
val = [(e["epoch"], e.get("val_acc"), e.get("val_loss")) for e in E if "val_acc" in e]

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))

ax_acc.plot(ep, train_acc, label="train_acc", alpha=.85)
if val:
    ax_acc.plot([x for x, _, _ in val], [a for _, a, _ in val], "o-", label="val_acc")
best = log.get("best_epoch")
if best is not None:
    ax_acc.axvline(best, ls="--", c="gray", alpha=.6, label=f"best @ep{best}")
ax_acc.set_xlabel("época"); ax_acc.set_ylabel("acurácia"); ax_acc.grid(alpha=.3); ax_acc.legend()

ax_loss.plot(ep, train_los, label="train_loss", alpha=.85)
vl = [(x, l) for x, _, l in val if l is not None]
if vl:
    ax_loss.plot([x for x, _ in vl], [l for _, l in vl], "o-", label="val_loss")
ax_loss.set_xlabel("época"); ax_loss.set_ylabel("loss"); ax_loss.grid(alpha=.3); ax_loss.legend()

fig.suptitle(f"{RUN_ID} — best_val_acc={log.get('best_val_acc')} @ época {log.get('best_epoch')}")
fig.tight_layout(); plt.show()

In [ ]:
from IPython.display import Image, display
from pathlib import Path
png = f"{OUTPUT_ROOT}/results/{RUN_ID}/confusion.png"
display(Image(filename=png)) if Path(png).exists() else print("confusion.png ainda não gerado")